# 05 — Feature Importance

**AFML Chapter 8** — López de Prado (2018)

Standard feature importance (MDI / Gini importance) is biased:
- Favours high-cardinality features
- Cannot distinguish signal from correlated noise
- Doesn't account for financial time-series contamination

This notebook compares three methods:
- **MDI** — Mean Decrease Impurity (fast, biased)
- **MDA** — Mean Decrease Accuracy (permutation, unbiased)
- **SFI** — Single Feature Importance (individual CV, slowest but cleanest)

We build features on BTC and show which methods correctly identify
signal vs noise, using purged CV throughout.

---

In [ ]:
from _setup import *
from afml.labeling import daily_volatility, make_events, triple_barrier_labels
from afml.sample_weights import sample_weight_by_return, normalize_weights
from afml.fracdiff import frac_diff_log
from afml.cross_validation import cv_score
from afml.feature_importance import (
    mean_decrease_impurity,
    mean_decrease_accuracy,
    single_feature_importance,
)
from validation.cv import PurgedKFold

from sklearn.ensemble import RandomForestClassifier

## 1. Build features (signal + noise)

In [ ]:
panel = load_daily_bars()
btc = panel[panel["symbol"] == "BTC-USD"].sort_values("ts").drop_duplicates("ts", keep="last")
close = btc.set_index("ts")["close"]

# Labels
vol = daily_volatility(close, span=20)
events = make_events(close, vol, holding_periods=10)
tb = triple_barrier_labels(close, events, pt_sl=(2.0, 2.0))
tb["y"] = (tb["label"] == 1).astype(int)

# Signal features (plausibly informative)
feat = pd.DataFrame(index=tb.index)
feat["ret_1d"] = np.log(close / close.shift(1)).reindex(tb.index)
feat["ret_5d"] = np.log(close / close.shift(5)).reindex(tb.index)
feat["ret_21d"] = np.log(close / close.shift(21)).reindex(tb.index)
feat["vol_20d"] = vol.reindex(tb.index)
feat["fracdiff"] = frac_diff_log(close, d=0.3).reindex(tb.index)

# Noise features (pure random — should score low)
rng = np.random.default_rng(42)
for i in range(3):
    feat[f"noise_{i}"] = rng.normal(size=len(feat))

feature_names = list(feat.columns)

# Align and clean
df = feat.join(tb[["y", "t1"]]).dropna()
X = df[feature_names].values
y = df["y"].values
t1 = pd.Series(df["t1"].values, index=df.index)
sw = normalize_weights(sample_weight_by_return(t1, close)).reindex(df.index).values

print(f"Samples: {len(df):,} | Features: {len(feature_names)} (5 signal + 3 noise)")
print(f"Feature names: {feature_names}")

## 2. MDI — Mean Decrease Impurity

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, n_jobs=-1)
rf.fit(X, y, sample_weight=sw)

mdi = mean_decrease_impurity(rf, feature_names)
display(mdi.style.format({"mdi_mean": "{:.3f}", "mdi_std": "{:.3f}"}).bar(subset=["mdi_mean"], color=NAVY))
print("\nNote: MDI is biased — noise features may get non-trivial importance")

## 3. MDA — Mean Decrease Accuracy

In [ ]:
cv = PurgedKFold(n_splits=5, t1=t1, pct_embargo=0.01)

mda = mean_decrease_accuracy(
    RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, n_jobs=-1),
    X, y, cv, feature_names,
    sample_weight=sw,
    n_repeats=3,
)
display(mda.style.format({"mda_mean": "{:.4f}", "mda_std": "{:.4f}"}).bar(subset=["mda_mean"], color=TEAL))
print("\nMDA is unbiased: noise features should show ~0 accuracy drop")

## 4. SFI — Single Feature Importance

In [ ]:
sfi = single_feature_importance(
    RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, n_jobs=-1),
    X, y, cv, feature_names,
    sample_weight=sw,
)
display(sfi.style.format({"sfi_mean": "{:.3f}", "sfi_std": "{:.3f}"}).bar(subset=["sfi_mean"], color=GOLD))
print("\nSFI: features trained individually. Noise should be ~50% (random)")

## 5. Comparison: all three methods

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, df_imp, col, title, color in zip(
    axes,
    [mdi, mda, sfi],
    ["mdi_mean", "mda_mean", "sfi_mean"],
    ["MDI (Gini Importance)", "MDA (Permutation)", "SFI (Single Feature CV)"],
    [NAVY, TEAL, GOLD],
):
    std_col = col.replace("mean", "std")
    # Sort by mean for each subplot
    df_plot = df_imp.sort_values(col, ascending=True)

    is_noise = df_plot["feature"].str.startswith("noise")
    bar_colors = [RED if n else color for n in is_noise]

    ax.barh(range(len(df_plot)), df_plot[col].values, xerr=df_plot[std_col].values,
            color=bar_colors, alpha=0.7, capsize=3)
    ax.set_yticks(range(len(df_plot)))
    ax.set_yticklabels(df_plot["feature"].values)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(col.split("_")[0].upper())

# Legend
from matplotlib.patches import Patch
axes[2].legend(
    [Patch(color=NAVY, alpha=0.7), Patch(color=RED, alpha=0.7)],
    ["Signal feature", "Noise feature"],
    loc="lower right", fontsize=9,
)

fig.suptitle("Feature Importance: MDI vs MDA vs SFI (BTC, purged CV)",
             fontweight="bold", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "05_feature_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Summary

**Key findings from applying AFML Chapter 8 to crypto:**

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| MDI | Fast, from tree structure | Biased towards high-cardinality; noise features get non-zero scores |
| MDA | Unbiased, works with any model | Slow (n_features × n_repeats × n_folds fits) |
| SFI | Cleanest signal, no substitution effects | Slowest; misses feature interactions |

**Recommendations:**
1. Use MDI for fast screening, but always verify with MDA/SFI
2. MDA is the default choice for production feature selection
3. SFI identifies which features have standalone predictive power
4. Always use purged CV + sample weights for all importance methods

---

*Next: [06_bet_sizing.ipynb](06_bet_sizing.ipynb) — Bet Sizing (AFML Ch 10)*